In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import torch

# Force CUDA re-initialisation
if torch.cuda.is_available():
    torch.cuda.init()

print(torch.cuda.device_count())  # should now be > 0

In [ ]:
import sys
import os
import pickle
import logging
import warnings
import re
from typing import Dict, List
from pathlib import Path
import json 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import geopandas as gpd

import torch

import massbalancemachine as mbm

sys.path.append(os.path.join(os.getcwd(), '../../'))
from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.device_count() == 0:
    print("No GPU detected. Using CPU.")
    device = torch.device("cpu")

In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / f"FD_MODEL_"
CACHE_DIR = BASE_DIR / "FD_MODEL_cache/"

# make sure BASE_DIR exists
os.makedirs(BASE_DIR, exist_ok=True)

# make sure BASE_DIR exists
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Base directory for data: {BASE_DIR}")
print(f"Cache directory for models: {CACHE_DIR}")

In [ ]:
MONTHLY_COLS = [
    't2m',
    'tp',
    'slhf',
    'sshf',
    'ssrd',
    'fal',
    'str',
    'ELEVATION_DIFFERENCE',
]
STATIC_COLS = ['aspect', 'slope', 'svf']

feature_columns = MONTHLY_COLS + STATIC_COLS

vois_climate = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
]

vois_topographical = ["aspect", "slope", "svf"]

paths_multi = {
    "EU_US_CANADA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_EU_US_CANADA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_EU_US_CANADA.nc"),
    },
    "HMA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_HIGH_MOUNTAIN_ASIA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_HIGH_MOUNTAIN_ASIA.nc"),
    },
}

## Datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (SJM+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
num_measurements

In [ ]:
# separate central asia into subregions
rgi_gdf = gpd.read_file(cfg.dataPath / RGI_V6_ROOT /
                        RGI_REGIONS['13']['folder'] /
                        RGI_REGIONS['13']['file'])[['RGIId', 'O2Region']]

glacier_dict_ca = (dfs['13'][['GLACIER', 'RGIId']].drop_duplicates().merge(
    rgi_gdf, on='RGIId', how='left').set_index('GLACIER').to_dict('index'))

dfs['13']['O2Region'] = dfs['13']['GLACIER'].map({
    k: v['O2Region']
    for k, v in glacier_dict_ca.items()
})

print(dfs['13'].groupby('O2Region').size().rename('n_measurements'))

# Build O2Region -> color mapping
o2_regions = sorted(
    set(v['O2Region'] for v in glacier_dict_ca.values()
        if v['O2Region'] is not None))
region_colors = [
    "red", "blue", "green", "purple", "orange", "darkred", "cadetblue",
    "darkgreen", "darkpurple", "pink", "gray", "black"
]
o2_color_map = {
    r: region_colors[i % len(region_colors)]
    for i, r in enumerate(o2_regions)
}

# Map each glacier to its O2Region color
color_map = {
    glacier: o2_color_map[info['O2Region']]
    for glacier, info in glacier_dict_ca.items()
    if info['O2Region'] is not None
}

m = plot_stakes_folium(dfs['13'], color_map=color_map, tooltip_col='O2Region')
m

In [ ]:
deduped = dfs['01'][['GLACIER', 'RGIId']].drop_duplicates()
multi_mask = deduped.groupby('GLACIER')['RGIId'].transform('count') > 1

deduped = deduped.copy()
deduped['GLACIER_NEW'] = deduped['GLACIER']
deduped.loc[multi_mask, 'GLACIER_NEW'] = (
    deduped.loc[multi_mask, 'GLACIER'] + '_' +
    deduped.loc[multi_mask, 'RGIId'].str.extract(r'\.(\d+)$')[0])
# EASTFORK_01 → EASTFORK_01_00022, EASTFORK_01_00021

dfs['01'] = dfs['01'].merge(deduped[['GLACIER', 'RGIId', 'GLACIER_NEW']],
                            on=['GLACIER', 'RGIId'],
                            how='left')
dfs['01']['GLACIER'] = dfs['01']['GLACIER_NEW'].fillna(dfs['01']['GLACIER'])
dfs['01'] = dfs['01'].drop(columns='GLACIER_NEW')

# separate Alaska into subregions
rgi_gdf = gpd.read_file(cfg.dataPath / RGI_V6_ROOT /
                        RGI_REGIONS['01']['folder'] /
                        RGI_REGIONS['01']['file'])[['RGIId', 'O2Region']]

glacier_dict_alaska = (dfs['01'][['GLACIER', 'RGIId']].drop_duplicates().merge(
    rgi_gdf, on='RGIId', how='left').set_index('GLACIER').to_dict('index'))

dfs['01']['O2Region'] = dfs['01']['GLACIER'].map({
    k: v['O2Region']
    for k, v in glacier_dict_alaska.items()
})

print(dfs['01'].groupby('O2Region').size().rename('n_measurements'))

# Build O2Region -> color mapping
o2_regions = sorted(
    set(v['O2Region'] for v in glacier_dict_alaska.values()
        if v['O2Region'] is not None))
o2_color_map = {
    r: region_colors[i % len(region_colors)]
    for i, r in enumerate(o2_regions)
}

# Map each glacier to its O2Region color
color_map = {
    glacier: o2_color_map[info['O2Region']]
    for glacier, info in glacier_dict_alaska.items()
    if info['O2Region'] is not None
}
m = plot_stakes_folium(dfs['01'], color_map=color_map, tooltip_col='O2Region')
m

In [ ]:
SOURCE_REGIONS = ['CH', 'NOR', 'ISL', "FR"]
TARGET_REGIONS = ['IT_AT', 'SJM', 'CENTRALASIA', 'ALA']

run_flag_by_code = {
    'ALA': False,
    'CENTRALASIA': False,
    'CH': False,
    'ISL': False,
    'NOR': False,
    'SJM': False,
    'FR': False,
    'IT_AT': False
}
# Step 2: compute monthly data once per code
monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=vois_climate,
    vois_topographical=vois_topographical,
    region_codes=SOURCE_REGIONS + TARGET_REGIONS,
    run_flag_by_code=run_flag_by_code,
)

In [ ]:
# Separate AT and IT glaciers from the combined dataset for easier handling in experiments
AT_GLACIERS = [
    'GOLDBERG K.',
    'HALLSTAETTER G.',
    'HINTEREIS F.',
    'JAMTAL F.',
    'KESSELWAND F.',
    'KLEINFLEISS K.',
    'OE. WURTEN K.',
    'VENEDIGER K.',
    'VERNAGT F.',
    'ZETTALUNITZ/MULLWITZ K.',
]

IT_GLACIERS = [
    'CAMPO SETT.',
    'CARESER',
    'CARESER CENTRALE',
    'CARESER OCCIDENTALE',
    'CARESER ORIENTALE',
    'CIARDONEY',
    'FONTANA BIANCA / WEISSBRUNNF.',
    'GRAND ETRET',
    'LUNGA (VEDRETTA) / LANGENF.',
    'LUPO',
    'MALAVALLE (VEDR. DI) / UEBELTALF.',
    'PENDENTE (VEDR.) / HANGENDERF.',
    'RIES OCC. (VEDR. DI) / RIESERF. WESTL.',
    'SURETTA MERIDIONALE',
]

data_AT = monthly_cache['IT_AT']['data_monthly'][monthly_cache['IT_AT'][
    'data_monthly']['GLACIER'].isin(AT_GLACIERS)].copy()
data_AT['SOURCE_CODE'] = 'AT'
data_AT_aug = monthly_cache['IT_AT']['data_monthly_aug'][monthly_cache[
    'IT_AT']['data_monthly_aug']['GLACIER'].isin(AT_GLACIERS)].copy()
data_AT_aug['SOURCE_CODE'] = 'AT'

data_IT = monthly_cache['IT_AT']['data_monthly'][monthly_cache['IT_AT'][
    'data_monthly']['GLACIER'].isin(IT_GLACIERS)].copy()
data_IT['SOURCE_CODE'] = 'IT'
data_IT_aug = monthly_cache['IT_AT']['data_monthly_aug'][monthly_cache[
    'IT_AT']['data_monthly_aug']['GLACIER'].isin(IT_GLACIERS)].copy()
data_IT_aug['SOURCE_CODE'] = 'IT'

# drop monthly_cache with 'IT_AT'
monthly_cache['IT'] = {
    'data_monthly': data_IT,
    'data_monthly_aug': data_IT_aug,
    'months_head_pad': monthly_cache['IT_AT']['months_head_pad'],
    'months_tail_pad': monthly_cache['IT_AT']['months_tail_pad']
}
monthly_cache['AT'] = {
    'data_monthly': data_AT,
    'data_monthly_aug': data_AT_aug,
    'months_head_pad': monthly_cache['IT_AT']['months_head_pad'],
    'months_tail_pad': monthly_cache['IT_AT']['months_tail_pad']
}
monthly_cache = {
    key: val
    for key, val in monthly_cache.items() if key != 'IT_AT'
}

print('Measurements per region after splitting IT and AT:')
print(f"Number of measurements in IT: {len(data_IT)}")
print(f"Number of measurements in AT: {len(data_AT)}")

In [ ]:
# --- CENTRAL ASIA subregions ---
data_ca = monthly_cache['CENTRALASIA']['data_monthly'].copy()
data_ca['O2Region'] = data_ca['GLACIER'].map({
    k: v['O2Region']
    for k, v in glacier_dict_ca.items()
})
data_ca_aug = monthly_cache['CENTRALASIA']['data_monthly_aug'].copy()
data_ca_aug['O2Region'] = data_ca_aug['GLACIER'].map({
    k: v['O2Region']
    for k, v in glacier_dict_ca.items()
})

data_CA_12 = data_ca[data_ca['O2Region'].isin(['1', '2'])].copy()
data_CA_3 = data_ca[data_ca['O2Region'] == '3'].copy()
data_CA_4 = data_ca[data_ca['O2Region'] == '4'].copy()
data_CA_12_aug = data_ca_aug[data_ca_aug['O2Region'].isin(['1', '2'])].copy()
data_CA_3_aug = data_ca_aug[data_ca_aug['O2Region'] == '3'].copy()
data_CA_4_aug = data_ca_aug[data_ca_aug['O2Region'] == '4'].copy()

for key, dm, dm_aug in [
    ('CA_12', data_CA_12, data_CA_12_aug),
    ('CA_3', data_CA_3, data_CA_3_aug),
    ('CA_4', data_CA_4, data_CA_4_aug),
]:
    dm['SOURCE_CODE'] = key
    dm_aug['SOURCE_CODE'] = key
    monthly_cache[key] = {
        'data_monthly': dm,
        'data_monthly_aug': dm_aug,
        'months_head_pad': monthly_cache['CENTRALASIA']['months_head_pad'],
        'months_tail_pad': monthly_cache['CENTRALASIA']['months_tail_pad'],
    }

print('Measurements per CENTRALASIA subregion:')
for key in ['CA_12', 'CA_3', 'CA_4']:
    print(f"  {key}: {len(monthly_cache[key]['data_monthly'])}")

# --- Alaska subregions ---
data_ala = monthly_cache['ALA']['data_monthly'].copy()
data_ala['O2Region'] = data_ala['GLACIER'].map({
    k: v['O2Region']
    for k, v in glacier_dict_alaska.items()
})
data_ala_aug = monthly_cache['ALA']['data_monthly_aug'].copy()
data_ala_aug['O2Region'] = data_ala_aug['GLACIER'].map({
    k: v['O2Region']
    for k, v in glacier_dict_alaska.items()
})

data_ALA_2 = data_ala[data_ala['O2Region'] == '2'].copy()
data_ALA_4 = data_ala[data_ala['O2Region'] == '4'].copy()
data_ALA_6 = data_ala[data_ala['O2Region'] == '6'].copy()
data_ALA_2_aug = data_ala_aug[data_ala_aug['O2Region'] == '2'].copy()
data_ALA_4_aug = data_ala_aug[data_ala_aug['O2Region'] == '4'].copy()
data_ALA_6_aug = data_ala_aug[data_ala_aug['O2Region'] == '6'].copy()

for key, dm, dm_aug in [
    ('ALA_2', data_ALA_2, data_ALA_2_aug),
    ('ALA_4', data_ALA_4, data_ALA_4_aug),
    ('ALA_6', data_ALA_6, data_ALA_6_aug),
]:
    dm['SOURCE_CODE'] = key
    dm_aug['SOURCE_CODE'] = key
    monthly_cache[key] = {
        'data_monthly': dm,
        'data_monthly_aug': dm_aug,
        'months_head_pad': monthly_cache['ALA']['months_head_pad'],
        'months_tail_pad': monthly_cache['ALA']['months_tail_pad'],
    }

print('\nMeasurements per Alaska subregion:')
for key in ['ALA_2', 'ALA_4', 'ALA_6']:
    print(f"  {key}: {len(monthly_cache[key]['data_monthly'])}")

In [ ]:
# With subregions
TARGET_REGIONS_SUB = [
    'IT', 'SJM', 'CENTRALASIA', 'ALA', 'CA_12', 'CA_3', 'CA_4', 'ALA_2',
    'ALA_4', 'ALA_6'
]

XREG_PAIRS = [(src,
               [r for r in SOURCE_REGIONS + TARGET_REGIONS_SUB if r != src])
              for src in SOURCE_REGIONS]

# Step 3: assemble train/test pairs from cache — no ERA5 reprocessing
res_xreg_by_source, split_info_by_source = prepare_xreg_pairs_from_cache(
    cfg=cfg,
    monthly_cache=monthly_cache,
    xreg_pairs=XREG_PAIRS,
)

In [ ]:
months_head_pad = res_xreg_by_source["CH"]['months_head_pad']
months_tail_pad = res_xreg_by_source["CH"]['months_tail_pad']
print(f"months_head_pad: {months_head_pad}")
print(f"months_tail_pad: {months_tail_pad}")

### Model assets:

#### Pooled set:

In [ ]:
# Build glacier→region mapping from all source regions (no ranking needed)
df_all_glaciers = pd.concat(
    [
        res_xreg_by_source[region]["df_train"][[
            "GLACIER"
        ]].drop_duplicates().assign(region=region) for region in SOURCE_REGIONS
    ],
    ignore_index=True,
).rename(columns={"GLACIER": "glacier"})

# Add measurement counts (needed by build_assets_from_glacier_list internally? No — only used by build_glacier_subsets)
# df_ranked just needs columns: "glacier", "region"  ← that's all build_assets uses
glaciers_all = df_all_glaciers["glacier"].tolist()
print(f"Total unique glaciers across all source regions: {len(glaciers_all)}")

assets_full = build_assets_from_glacier_list(
    glaciers=glaciers_all,
    df_ranked=df_all_glaciers,  # acts as the gl→region lookup
    res_xreg_by_source=res_xreg_by_source,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    cfg=cfg,
    cache_path=str(CACHE_DIR / "assets_foundation_100pct.joblib"),
    force_recompute=True,
    months_head_pad=months_head_pad,
    months_tail_pad=months_tail_pad,
    src_regions=SOURCE_REGIONS,
)

# Failsafe: check that pooled training data only contains source region codes
actual_codes = set(df_all_glaciers.region.unique())
expected_codes = set(SOURCE_REGIONS)
unexpected = actual_codes - expected_codes
if unexpected:
    raise ValueError(
        f"Pooled training set contains unexpected SOURCE_CODEs: {unexpected}")
else:
    print(f"✓ Pooled set SOURCE_CODEs OK: {actual_codes}")

print(f"Foundation model assets: {len(assets_full['ds_train'])} sequences")

#### Test regions:

In [ ]:
# ── Always compute scalers/blurs — needed for any recomputation ───────────────
res_source_only = {
    region: {
        "df_train": res_xreg_by_source[region]["df_train"],
    }
    for region in SOURCE_REGIONS
}

scaler_m, scaler_s, scaler_joint = build_global_scalers_multi_source_simple(
    res_source_only,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
)
blur_m, blur_s, blur_joint = estimate_global_bandwidths_simple(
    res_source_only,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    scaler_m=scaler_m,
    scaler_s=scaler_s,
    seed=cfg.seed,
)
print(
    f"Estimated blurs: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}"
)

In [ ]:
RECOMPUTE_SPLITS = False
FORCE_RECOMPUTE = False
FORCE_RECOMPUTE_REGIONS = ['ALA_2', 'ALA_4', 'ALA_6']

FT_FRAC = 0.10
save_path = BASE_DIR / "glacier_splits_FD.json"

gl_level_split = ['CENTRALASIA', 'IT', 'ALA']
id_level_split = [
    'SJM', 'CENTRALASIA', 'ALA', 'IT', 'CA_12', 'CA_3', 'CA_4', 'ALA_2',
    'ALA_4', 'ALA_6'
]

# ── Load existing glacier splits from disk ────────────────────────────────────
splits = {}
if not RECOMPUTE_SPLITS and save_path.exists():
    with open(save_path, "r") as f:
        splits = json.load(f)
    print(f"Loaded splits from: {save_path}")

# ── Only compute glacier splits for regions that need them ────────────────────
regions_needing_gl_split = [r for r in gl_level_split]
regions_to_compute = (regions_needing_gl_split if RECOMPUTE_SPLITS else
                      [r for r in regions_needing_gl_split if r not in splits])

for region in regions_to_compute:
    print(f"\n{'='*50}\nSplitting region: {region}")
    split = split_pool_holdout_sinkhorn(
        df_region=monthly_cache[region]["data_monthly"],
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        scaler_m=scaler_m,
        scaler_s=scaler_s,
        blur_joint=blur_joint,
        pool_frac=0.2,
        n_restarts=50,
        seed=cfg.seed,
    )
    splits[region] = split
    print(f"  Pool glaciers    : {split['pool_glaciers']}")
    print(f"  Holdout glaciers : {split['holdout_glaciers']}")
    print(f"  Pool fraction    : {split['actual_pool_frac']:.1%}")
    print(f"  Sk(pool↔holdout) : {split['sinkhorn_pool_vs_holdout']:.4f}")

if regions_to_compute:
    with open(save_path, "w") as f:
        json.dump(splits, f, indent=2)
    print(f"\nSaved splits for: {regions_to_compute}")

# ── Build FT assets for all target subregions ─────────────────────────────────
ft_assets_glacier = {}
ft_assets_id = {}

for region in TARGET_REGIONS_SUB:
    print(f"\n{'='*50}\nBuilding FT assets for: {region}")

    df_monthly = monthly_cache[region]["data_monthly"]
    df_monthly_aug = monthly_cache[region]["data_monthly_aug"]
    head_pad = monthly_cache[region]["months_head_pad"]
    tail_pad = monthly_cache[region]["months_tail_pad"]
    force = FORCE_RECOMPUTE or (region in FORCE_RECOMPUTE_REGIONS)

    # 1. Glacier-level split
    if region in gl_level_split:
        pool_glaciers = splits[region]["pool_glaciers"]
        holdout_glaciers = splits[region]["holdout_glaciers"]
        exp_key = f"FD_to_{region}"

        ds_ft_gl = build_or_load_lstm_dataset_only(
            cfg=cfg,
            key=f"{exp_key}_FT",
            df_loss=df_monthly[df_monthly["GLACIER"].isin(pool_glaciers)],
            df_full=df_monthly_aug[df_monthly_aug["GLACIER"].isin(
                pool_glaciers)],
            months_head_pad=head_pad,
            months_tail_pad=tail_pad,
            MONTHLY_COLS=MONTHLY_COLS,
            STATIC_COLS=STATIC_COLS,
            cache_dir=str(CACHE_DIR),
            force_recompute=force,
            kind="ft",
            show_progress=False,
        )
        ds_test_gl = build_or_load_lstm_dataset_only(
            cfg=cfg,
            key=f"{exp_key}_TEST",
            df_loss=df_monthly[df_monthly["GLACIER"].isin(holdout_glaciers)],
            df_full=df_monthly_aug[df_monthly_aug["GLACIER"].isin(
                holdout_glaciers)],
            months_head_pad=head_pad,
            months_tail_pad=tail_pad,
            MONTHLY_COLS=MONTHLY_COLS,
            STATIC_COLS=STATIC_COLS,
            cache_dir=str(CACHE_DIR),
            force_recompute=force,
            kind="test",
            show_progress=False,
        )
        ft_train_idx_gl, ft_val_idx_gl = mbm.data_processing.MBSequenceDataset.split_indices(
            len(ds_ft_gl), val_ratio=0.2, seed=cfg.seed)
        ft_assets_glacier[region] = {
            "ds_ft": ds_ft_gl,
            "ds_test": ds_test_gl,
            "ft_train_idx": ft_train_idx_gl,
            "ft_val_idx": ft_val_idx_gl,
        }
        print(
            f"  [glacier-split] ds_ft  : {len(ds_ft_gl)} seqs | ds_test: {len(ds_test_gl)} seqs"
        )

    # 2. ID-level split
    if region in id_level_split:
        winter_ids = df_monthly[df_monthly["PERIOD"] ==
                                "winter"]["ID"].unique().tolist()
        annual_ids = df_monthly[df_monthly["PERIOD"] ==
                                "annual"]["ID"].unique().tolist()

        rng = np.random.default_rng(cfg.seed)
        rng.shuffle(winter_ids)
        rng.shuffle(annual_ids)

        n_ft_w = max(1, int(len(winter_ids) * FT_FRAC)) if winter_ids else 0
        n_ft_a = max(1, int(len(annual_ids) * FT_FRAC))
        ft_ids = winter_ids[:n_ft_w] + annual_ids[:n_ft_a]
        test_ids = winter_ids[n_ft_w:] + annual_ids[n_ft_a:]

        exp_key_id = f"FD_to_{region}_IDsplit"
        ds_ft_id = build_or_load_lstm_dataset_only(
            cfg=cfg,
            key=f"{exp_key_id}_FT",
            df_loss=df_monthly[df_monthly["ID"].isin(ft_ids)],
            df_full=df_monthly_aug[df_monthly_aug["ID"].isin(ft_ids)],
            months_head_pad=head_pad,
            months_tail_pad=tail_pad,
            MONTHLY_COLS=MONTHLY_COLS,
            STATIC_COLS=STATIC_COLS,
            cache_dir=str(CACHE_DIR),
            force_recompute=force,
            kind="ft",
            show_progress=False,
        )
        ds_test_id = build_or_load_lstm_dataset_only(
            cfg=cfg,
            key=f"{exp_key_id}_TEST",
            df_loss=df_monthly[df_monthly["ID"].isin(test_ids)],
            df_full=df_monthly_aug[df_monthly_aug["ID"].isin(test_ids)],
            months_head_pad=head_pad,
            months_tail_pad=tail_pad,
            MONTHLY_COLS=MONTHLY_COLS,
            STATIC_COLS=STATIC_COLS,
            cache_dir=str(CACHE_DIR),
            force_recompute=force,
            kind="test",
            show_progress=False,
        )
        ft_train_idx_id, ft_val_idx_id = mbm.data_processing.MBSequenceDataset.split_indices(
            len(ds_ft_id), val_ratio=0.2, seed=cfg.seed)
        ft_assets_id[region] = {
            "ds_ft": ds_ft_id,
            "ds_test": ds_test_id,
            "ft_train_idx": ft_train_idx_id,
            "ft_val_idx": ft_val_idx_id,
        }
        print(
            f"  [ID-split]      ds_ft  : {len(ds_ft_id)} seqs | ds_test: {len(ds_test_id)} seqs"
        )

# convenience dict: glacier-split where available, ID-split for the rest
ft_assets = {
    **ft_assets_glacier,
    **{
        r: ft_assets_id[r]
        for r in ft_assets_id if r not in ft_assets_glacier
    },
}

## Transformer:

In [ ]:
best_params_gs = {
    'Fm': 8,
    'Fs': 3,
    'd_model': 128,
    'nhead': 8,
    'num_layers': 3,
    'dim_feedforward': 128,
    'dropout': 0.1,
    'static_layers': 2,
    'static_hidden': 64,
    'static_dropout': 0.0,
    'head_dropout': 0.1,
    'lr': 0.0005,
    'weight_decay': 1e-05,
    'two_heads': False,
    'loss_spec': None,
    'T_max': 32
}

print("\nbest_params_gs:")
for k, v in best_params_gs.items():
    print(f"  {k}: {v}")

TRAIN_FLAG = True
MODEL_DATE = "2026-06-10"
models_dir = BASE_DIR / "models" / "foundation"
os.makedirs(models_dir, exist_ok=True)

model_foundation, model_path, info = train_or_load_one_source_model(
    cfg=cfg,
    key="foundation_100pct",
    lstm_assets=assets_full,
    best_params=best_params_gs,
    device=device,
    models_dir=models_dir,
    prefix="transformer_foundation",
    train_flag=TRAIN_FLAG,
    force_retrain=True,
    epochs=150,
    date=MODEL_DATE,
    model_type="transformer",
    verbose=True,
)

print(f"Foundation model saved to: {model_path}")

In [ ]:
# Build scaler once outside the loop
ds_pooled_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
    assets_full["ds_train"])
ds_pooled_scaler.make_loaders(
    train_idx=assets_full["train_idx"],
    val_idx=assets_full["val_idx"],
    fit_and_transform=True,
    seed=cfg.seed,
    verbose=False,
)

print(f"\n{'='*60}")
print("Zero-shot evaluation of foundation model on target regions")
print(f"{'='*60}")

for split_label, assets_dict, regions in [
    ("glacier-split", ft_assets_glacier, gl_level_split),
    ("ID-split", ft_assets_id, id_level_split),
]:
    print(f"\n--- {split_label} ---")
    for region in regions:
        ds_test_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
            assets_dict[region]["ds_test"])
        test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
            ds_test_copy, ds_pooled_scaler, batch_size=128, seed=cfg.seed)
        metrics, _ = model_foundation.evaluate_with_preds(
            device, test_dl, ds_test_copy)
        print(f"  {region:15s}  RMSE_a={metrics['RMSE_annual']:.3f}  "
              f"RMSE_w={metrics['RMSE_winter']:.3f}  "
              f"R2_a={metrics['R2_annual']:.3f}  "
              f"R2_w={metrics['R2_winter']:.3f}")

In [ ]:
for split_label, assets_dict, regions in [
    ("glacier-split", ft_assets_glacier, gl_level_split),
    ("ID-split", ft_assets_id, id_level_split),
]:
    for region in regions:
        plot_pred_vs_truth_grid(
            plot_configs=[(f"{region} (zero-shot)", model_foundation,
                           assets_full)],
            ds_test=assets_dict[region]["ds_test"],
            device=device,
            cfg=cfg,
            ncols=1,
            ax_xlim=(-8, 6),
            ax_ylim=(-8, 6),
            title=f"Foundation model → {region} (zero-shot, {split_label})",
            save_path=f"figures/foundation_zeroshot_{region}_{split_label}",
            figsize=(5, 5),
        )

#### Compare to single shot models:

In [ ]:
XREG_MODEL_DATE = "2026-06-10"
XREG_MODELS_BASE = Path(cfg.dataPath) / path_cache / "CROSS_REGION_/models"
TRAIN_FLAG = True
single_src_params = {
    'Fm': 8,
    'Fs': 3,
    'd_model': 64,
    'nhead': 8,
    'num_layers': 3,
    'dim_feedforward': 128,
    'dropout': 0.1,
    'static_layers': 1,
    'static_hidden': 128,
    'static_dropout': 0.1,
    'head_dropout': 0.1,
    'lr': 0.0005,
    'weight_decay': 1e-05,
    'two_heads': False,
    'loss_spec': None,
    'T_max': 32
}

single_source_models = {}
single_source_assets = {}
single_source_scalers = {}

for src in SOURCE_REGIONS:
    ckpt = XREG_MODELS_BASE / src / f"transformer_xreg_{src}_{src}_{XREG_MODEL_DATE}.pt"
    os.makedirs(ckpt.parent, exist_ok=True)

    df_src = res_xreg_by_source[src]["df_train"][["GLACIER"]] \
               .drop_duplicates().assign(region=src) \
               .rename(columns={"GLACIER": "glacier"})
    assets_src = build_assets_from_glacier_list(
        glaciers=df_src["glacier"].tolist(),
        df_ranked=df_src,
        res_xreg_by_source=res_xreg_by_source,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        cfg=cfg,
        cache_path=str(XREG_MODELS_BASE / src / f"assets_{src}_100pct.joblib"),
        force_recompute=True,
        months_head_pad=months_head_pad,
        months_tail_pad=months_tail_pad,
        src_regions=[src],
    )
    single_source_assets[src] = assets_src

    # build per-source scaler
    scaler_src = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        assets_src["ds_train"])
    scaler_src.make_loaders(
        train_idx=assets_src["train_idx"],
        val_idx=assets_src["val_idx"],
        fit_and_transform=True,
        seed=cfg.seed,
        verbose=False,
    )
    single_source_scalers[src] = scaler_src

    m = mbm.models.Transformer_MB.build_model_from_params(cfg,
                                                          single_src_params,
                                                          device,
                                                          verbose=False)
    if ckpt.exists():
        state = torch.load(ckpt, map_location=device)
        m.load_state_dict(state)
        print(f"Loaded {src} model from {ckpt}")
    else:
        print(f"No checkpoint found for {src}, training from scratch...")
        m, _, _ = train_or_load_one_source_model(
            cfg=cfg,
            key=src,
            lstm_assets=assets_src,
            best_params=single_src_params,
            device=device,
            models_dir=XREG_MODELS_BASE / src,
            prefix="transformer_xreg",
            train_flag=TRAIN_FLAG,
            force_retrain=True,
            epochs=150,
            date=XREG_MODEL_DATE,
            model_type="transformer",
            verbose=False,
        )
        print(f"Trained {src} model, saved to {ckpt}")

    single_source_models[src] = m

In [ ]:
# --- collect metrics ---
eval_configs = [
    ("IT (GL-split)", ft_assets_glacier.get("IT", ft_assets_id.get("IT"))),
    ("SJM (ID-split)", ft_assets_id["SJM"]),
    ("CENTRALASIA (GL-split)", ft_assets_glacier["CENTRALASIA"]),
    ("CENTRALASIA (ID-split)", ft_assets_id["CENTRALASIA"]),
    ("ALA (GL-split)", ft_assets_glacier.get("ALA", ft_assets_id.get("ALA"))),
    ("ALA (ID-split)", ft_assets_id.get("ALA")),
    ("ALA_2 (ID-split)", ft_assets_id.get("ALA_2")),
    ("ALA_4 (ID-split)", ft_assets_id.get("ALA_4")),
    ("ALA_6 (ID-split)", ft_assets_id.get("ALA_6")),
    ("CA_12 (ID-split)", ft_assets_id.get("CA_12")),
    ("CA_3 (ID-split)", ft_assets_id.get("CA_3")),
    ("CA_4 (ID-split)", ft_assets_id.get("CA_4")),
]


def _eval_on(model, ds_test, scaler):
    ds_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ds_test)
    test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
        ds_copy, scaler, batch_size=128, seed=cfg.seed)
    return model.evaluate_with_preds(device, test_dl, ds_copy)


df_metrics_comparison = {}
for tgt_label, assets in eval_configs:
    if assets is None:
        print(f"No assets for {tgt_label}, skipping.")
        continue

    df_metrics_comparison[tgt_label] = {}
    m, _ = _eval_on(model_foundation, assets["ds_test"], ds_pooled_scaler)
    df_metrics_comparison[tgt_label]["foundation"] = m

    for src, model in single_source_models.items():
        m, _ = _eval_on(model, assets["ds_test"], single_source_scalers[src])
        df_metrics_comparison[tgt_label][src] = m

# --- print summary ---
print(f"\n{'='*80}")
print(f"{'Model':<15} {'RMSE_a':>8} {'RMSE_w':>8} {'R2_a':>8} {'R2_w':>8}")
print(f"{'='*80}")
for tgt_label, results in df_metrics_comparison.items():
    print(f"\n--- {tgt_label} ---")
    for model_label, m in results.items():
        print(f"  {model_label:<15}  "
              f"RMSE_a={m['RMSE_annual']:.3f}  "
              f"RMSE_w={m['RMSE_winter']:.3f}  "
              f"R2_a={m['R2_annual']:.3f}  "
              f"R2_w={m['R2_winter']:.3f}")

In [ ]:
# --- bar plot ---
MODEL_ORDER = SOURCE_REGIONS + ["foundation"]
COLORS = {
    "CH": NATURE_PALETTE["blue"],
    "NOR": NATURE_PALETTE["vermillion"],
    "ISL": NATURE_PALETTE["reddish_purple"],
    "FR": NATURE_PALETTE["orange"],
    "foundation": "black",
}
LABELS = {src: f"{src} only" for src in SOURCE_REGIONS}
LABELS["foundation"] = "Foundation\n(pooled)"
METRICS = [
    ("RMSE_annual", "RMSE annual (m w.e.)", False),
    #("RMSE_winter", "RMSE winter (m w.e.)", False),
]


def has_winter_comparison(tgt_label):
    results = df_metrics_comparison[tgt_label]
    winter_vals = [
        results[m]["RMSE_winter"] for m in MODEL_ORDER
        if m in results and "RMSE_winter" in results[m]
    ]
    return any(not np.isnan(v) for v in winter_vals)


def get_metrics_for_region_comparison(tgt_label):
    if not has_winter_comparison(tgt_label):
        return [(k, l, r) for k, l, r in METRICS if k != "RMSE_winter"]
    return METRICS


n_tgts = len(df_metrics_comparison)
n_metrics = len(METRICS)

fig, axes = plt.subplots(
    n_metrics,
    n_tgts,
    figsize=(3.5 * n_tgts, 3.8 * n_metrics),
    sharey=False,
)
axes = np.array(axes).reshape(n_metrics, n_tgts)

for col, (tgt_label, results) in enumerate(df_metrics_comparison.items()):
    region_metrics = get_metrics_for_region_comparison(tgt_label)
    for row, (metric_key, metric_label, is_r2) in enumerate(METRICS):
        ax = axes[row, col]

        if (metric_key, metric_label, is_r2) not in region_metrics:
            ax.set_visible(False)
            continue

        models = [m for m in MODEL_ORDER if m in results]
        vals = [results[m][metric_key] for m in models]
        colors = [COLORS[m] for m in models]
        bars = ax.bar(range(len(models)),
                      vals,
                      color=colors,
                      edgecolor="white",
                      linewidth=0.8,
                      alpha=0.5,
                      zorder=3)

        fidx = models.index("foundation") if "foundation" in models else None
        if fidx is not None:
            bars[fidx].set_edgecolor("black")
            bars[fidx].set_linewidth(2)
            bars[fidx].set_alpha(0.5)

        for bar, val in zip(bars, vals):
            y_pos = bar.get_height() + 0.02 if val >= 0 else bar.get_height(
            ) - 0.08
            ax.text(bar.get_x() + bar.get_width() / 2,
                    y_pos,
                    f"{val:.2f}",
                    ha="center",
                    va="bottom",
                    fontsize=7)

        if is_r2:
            ax.axhline(0,
                       color="black",
                       linewidth=0.8,
                       linestyle="--",
                       zorder=2)

        ax.set_xticks(range(len(models)))
        if row == n_metrics - 1:
            ax.set_xticklabels([LABELS[m] for m in models],
                               rotation=30,
                               ha="right",
                               fontsize=NATURE_SPECS["font_max_pt"])
        else:
            ax.set_xticklabels([])

        if col == 0:
            ax.set_ylabel(metric_label, fontsize=NATURE_SPECS["font_max_pt"])
        if row == 0:
            ax.set_title(tgt_label,
                         fontsize=NATURE_SPECS["font_max_pt"] + 1,
                         fontweight="bold")

        valid_vals = [v for v in vals if not np.isnan(v)]
        if is_r2:
            ymin = min(min(valid_vals) * 1.3, -0.2)
            ymax = max(max(valid_vals) * 1.3, 0.5)
        else:
            ymin = 0
            ymax = max(valid_vals) * 1.25
        ax.set_ylim(ymin, ymax)

        ax.yaxis.grid(False, linestyle="--", alpha=0.5, zorder=0)
        ax.set_axisbelow(False)
        apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=False)

fig.suptitle("Foundation model (zero-shot) vs single-source Transformers",
             fontsize=NATURE_SPECS["font_max_pt"] + 2,
             fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.99])
fig.savefig("figures/foundation_vs_single_source_barplot.png",
            dpi=NATURE_SPECS["dpi"],
            bbox_inches="tight")
plt.show()

### Fine tuning:

In [ ]:
TRAIN_FLAG = False
models_dir_ft = BASE_DIR / "models" / "finetuned"
os.makedirs(models_dir_ft, exist_ok=True)

# scaler donor: always cloned from the foundation training assets
ds_pooled_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
    assets_full["ds_train"])
ds_pooled_scaler.make_loaders(
    train_idx=assets_full["train_idx"],
    val_idx=assets_full["val_idx"],
    fit_and_transform=True,
    seed=cfg.seed,
    verbose=False,
)

ft_models_gl = {}  # glacier-split
ft_models_id = {}  # ID-split only

for split_label, assets_dict, models_dict, regions, suffix in [
    ("GLsplit", ft_assets_glacier, ft_models_gl, gl_level_split, ""),
    ("IDsplit", ft_assets_id, ft_models_id, id_level_split, "_IDsplit"),
]:
    for region in regions:
        models_dict[region] = {}
        print(f"\n{'='*60}")
        print(f"Fine-tuning on: {region} ({split_label})")

        for strategy in ["head_only", "full", "adapter"]:
            models_dict[region][strategy] = finetune_transformer_on_target(
                cfg=cfg,
                model_pooled=model_foundation,
                ds_ft=assets_dict[region]["ds_ft"],
                ds_pooled_scaler=ds_pooled_scaler,
                device=device,
                best_params=best_params_gs,
                model_filename=str(
                    models_dir_ft /
                    f"transformer_ft_{strategy}_{region}{suffix}_{MODEL_DATE}.pt"
                ),
                strategy=strategy,
                epochs=50,
                lr_factor=0.1,
                force_retrain=TRAIN_FLAG,
            )
            print(f"  [{region}] {strategy} done")

In [ ]:
source_codes_pretrain = build_source_codes_for_dataset(
    assets_full["ds_train"],
    pd.concat(
        [res_xreg_by_source[r]["data_monthly_aug"] for r in SOURCE_REGIONS],
        ignore_index=False),
    source_col="SOURCE_CODE",
)

dan_models = {}  # glacier-split
dan_models_id = {}  # ID-split

for split_label, assets_dict, dan_dict, regions, suffix in [
    ("GLsplit", ft_assets_glacier, dan_models, gl_level_split, ""),
    ("IDsplit", ft_assets_id, dan_models_id, id_level_split, "_IDsplit"),
]:
    for region in regions:
        print(f"\n{'='*50}  DAN → {region} ({split_label})")

        source_codes_ft = [region] * len(assets_dict[region]["ds_ft"])

        dan_dict[region], _ = finetune_dan_transformer_on_target(
            cfg=cfg,
            model_foundation=model_foundation,
            assets_full=assets_full,
            ft_assets_region=assets_dict[region],
            ds_pooled_scaler=ds_pooled_scaler,
            source_codes_pretrain=source_codes_pretrain,
            source_codes_ft=source_codes_ft,
            device=device,
            best_params=best_params_gs,
            model_filename=str(
                models_dir_ft /
                f"transformer_dan_{region}{suffix}_{MODEL_DATE}.pt"),
            dan_alpha=0.1,
            grl_lambda=1.0,
            mix_ratio_ft=1.0,
            epochs=60,
            lr_backbone=5e-5,
            lr_domain=1e-4,
            force_retrain=TRAIN_FLAG,
            verbose=False,
        )

In [ ]:
# --- unified evaluation ---
print(f"\n{'='*75}")
print(f"{'Strategy':<25} {'RMSE_a':>8} {'RMSE_w':>8} {'R2_a':>8} {'R2_w':>8}")
print(f"{'='*75}")

for split_label, assets_dict, models_dict, dan_dict, regions in [
    ("default", ft_assets, ft_models_gl, dan_models, gl_level_split),
    ("IDsplit", ft_assets_id, ft_models_id, dan_models_id, id_level_split),
]:
    for region in regions:
        print(f"\n--- {region} ({split_label}) ---")

        def _eval(model, region=region, assets_dict=assets_dict):
            ds_test_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
                assets_dict[region]["ds_test"])
            test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
                ds_test_copy, ds_pooled_scaler, batch_size=128, seed=cfg.seed)
            return model.evaluate_with_preds(device, test_dl, ds_test_copy)

        all_models = {
            "no_ft": model_foundation,
            **{
                s: models_dict[region][s]
                for s in ["head_only", "adapter", "full"]
            },
            "dan": dan_dict[region],
        }

        for label, model in all_models.items():
            m, _ = _eval(model)
            print(f"  {label:<25}  "
                  f"RMSE_a={m['RMSE_annual']:.3f}  "
                  f"RMSE_w={m['RMSE_winter']:.3f}  "
                  f"R2_a={m['R2_annual']:.3f}  "
                  f"R2_w={m['R2_winter']:.3f}")

In [ ]:
for split_label, assets_dict, models_dict, dan_dict, regions in [
    ("GLsplit", ft_assets, ft_models_gl, dan_models, gl_level_split),
    ("IDsplit", ft_assets_id, ft_models_id, dan_models_id, id_level_split),
]:
    for region in regions:
        plot_configs = [
            ("no_ft (zero-shot)", model_foundation, assets_full),
            *[(strategy, models_dict[region][strategy], assets_full)
              for strategy in ["head_only", "full", "adapter"]],
            ("dan", dan_dict[region], assets_full),
        ]

        plot_pred_vs_truth_grid(
            plot_configs=plot_configs,
            ds_test=assets_dict[region]["ds_test"],
            device=device,
            cfg=cfg,
            ncols=5,
            ax_xlim=(-8, 6),
            ax_ylim=(-8, 6),
            title=
            f"Foundation model → {region}: zero-shot vs fine-tuning strategies ({split_label})",
            save_path=
            f"figures/foundation_ft_strategies_{region}_{split_label}",
            figsize=(25, 5),
        )

In [ ]:
for split_label, assets_dict, models_dict, dan_dict, regions in [
        # ("GLsplit", ft_assets, ft_models_gl, dan_models, gl_level_split),
    ("IDsplit", ft_assets_id, ft_models_id, dan_models_id, id_level_split),
]:
    for region in regions:
        plot_configs = [
            ("no_ft (zero-shot)", model_foundation, assets_full),
            *[(strategy, models_dict[region][strategy], assets_full)
              for strategy in ["full"]],
            ("dan", dan_dict[region], assets_full),
        ]

        plot_pred_vs_truth_grid(
            plot_configs=plot_configs,
            ds_test=assets_dict[region]["ds_test"],
            device=device,
            ncols=3,
            cfg=cfg,
            ax_xlim=(-8, 6),
            ax_ylim=(-8, 6),
            title=
            f"Foundation model → {region}: zero-shot vs fine-tuning strategies ({split_label})",
            save_path=
            f"figures/foundation_ft_strategies_{region}_{split_label}",
            figsize=(25, 5),
        )

##### Compare to single shot:

In [ ]:
# ================================================================
# Compare foundation FT vs single-source FT on IT, SJM, CENTRALASIA
# ================================================================

FT_STRATEGY = "full"  # toggle: "head_only", "adapter", "full"

XREG_FT_DATE = "2026-05-29"
XREG_MODELS_BASE = Path(cfg.dataPath) / path_cache / "CROSS_REGION" / "models"

# --- build single-source assets and scalers (needed for FT) ---
single_source_assets = {}
single_source_scalers = {}

for src in SOURCE_REGIONS:
    df_src = res_xreg_by_source[src]["data_monthly"][["GLACIER"]] \
               .drop_duplicates().assign(region=src) \
               .rename(columns={"GLACIER": "glacier"})
    assets_src = build_assets_from_glacier_list(
        glaciers=df_src["glacier"].tolist(),
        df_ranked=df_src,
        res_xreg_by_source=res_xreg_by_source,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        cfg=cfg,
        cache_path=str(XREG_MODELS_BASE / src / f"assets_{src}_100pct.joblib"),
        force_recompute=True,
        months_head_pad=months_head_pad,
        months_tail_pad=months_tail_pad,
        src_regions=[src],
    )
    single_source_assets[src] = assets_src

    scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        assets_src["ds_train"])
    scaler.make_loaders(
        train_idx=assets_src["train_idx"],
        val_idx=assets_src["val_idx"],
        fit_and_transform=True,
        seed=cfg.seed,
        verbose=False,
    )
    single_source_scalers[src] = scaler

# --- fine-tune each single-source model on each target ---
single_ft_models = {}  # single_ft_models[src][tgt_label]

for src in SOURCE_REGIONS:
    single_ft_models[src] = {}
    base_model = single_source_models[src]

    for tgt_label, assets in eval_configs:
        if assets is None:
            continue

        ft_dir = XREG_MODELS_BASE / src / "finetuned"
        os.makedirs(ft_dir, exist_ok=True)

        ckpt = ft_dir / f"transformer_ft_{FT_STRATEGY}_{src}_to_{tgt_label.replace(' ', '_').replace('(', '').replace(')', '')}_{XREG_FT_DATE}.pt"

        model_ft = finetune_transformer_on_target(
            cfg=cfg,
            model_pooled=base_model,
            ds_ft=assets["ds_ft"],
            ds_pooled_scaler=single_source_scalers[src],
            device=device,
            best_params=single_src_params,
            model_filename=str(ckpt),
            strategy=FT_STRATEGY,
            epochs=50,
            lr_factor=0.1,
            force_retrain=not ckpt.exists(),
        )
        single_ft_models[src][tgt_label] = model_ft
        print(f"  [{src} → {tgt_label}] {FT_STRATEGY} done")

In [ ]:
# --- collect metrics: foundation FT vs single-source FT ---
df_metrics_ft_comparison = {}

for tgt_label, assets in eval_configs:
    if assets is None:
        continue

    df_metrics_ft_comparison[tgt_label] = {}
    tgt_key = tgt_label.split()[0]
    is_id = "ID-split" in tgt_label or tgt_key == "SJM"  # SJM always ID-split
    mdict = ft_models_id if is_id else ft_models_gl

    # zero-shot foundation baseline
    m, _ = _eval_on(model_foundation, assets["ds_test"])
    df_metrics_ft_comparison[tgt_label]["foundation_zs"] = m

    # foundation fine-tuned
    if tgt_key in mdict and FT_STRATEGY in mdict[tgt_key]:
        m, _ = _eval_on(mdict[tgt_key][FT_STRATEGY], assets["ds_test"])
        df_metrics_ft_comparison[tgt_label]["foundation_ft"] = m

    # single-source FT
    for src in SOURCE_REGIONS:
        if tgt_label in single_ft_models.get(src, {}):
            m, _ = _eval_on(single_ft_models[src][tgt_label],
                            assets["ds_test"])
            df_metrics_ft_comparison[tgt_label][src] = m

# --- print summary ---
print(f"\n{'='*80}")
print(f"Strategy: {FT_STRATEGY}")
print(f"{'Model':<20} {'RMSE_a':>8} {'RMSE_w':>8} {'R2_a':>8} {'R2_w':>8}")
print(f"{'='*80}")
for tgt_label, results in df_metrics_ft_comparison.items():
    print(f"\n--- {tgt_label} ---")
    for model_label, m in results.items():
        print(f"  {model_label:<20}  "
              f"RMSE_a={m['RMSE_annual']:.3f}  "
              f"RMSE_w={m['RMSE_winter']:.3f}  "
              f"R2_a={m['R2_annual']:.3f}  "
              f"R2_w={m['R2_winter']:.3f}")

In [ ]:
# --- bar plot ---
FT_MODEL_ORDER = ["foundation_zs", "foundation_ft"] + SOURCE_REGIONS
FT_COLORS = {
    "foundation_zs": "#aaaaaa",
    "foundation_ft": "black",
    "CH": NATURE_PALETTE["blue"],
    "NOR": NATURE_PALETTE["vermillion"],
    "ISL": NATURE_PALETTE["reddish_purple"],
    "FR": NATURE_PALETTE["orange"],
}
FT_LABELS = {
    "foundation_zs": "Fnd.\n(0-shot)",
    "foundation_ft": f"Fnd.\n({FT_STRATEGY})",
    **{
        src: f"{src}\n({FT_STRATEGY})"
        for src in SOURCE_REGIONS
    },
}

METRICS = [
    ("RMSE_annual", "RMSE annual\n(m w.e.)", False),
    ("RMSE_winter", "RMSE winter\n(m w.e.)", False),
]

R2_YMIN_CLIP = -2.0


def has_winter(tgt_label):
    """Check whether a target region has any winter measurements."""
    # df_metrics_ft_comparison keys are display labels, not region codes.
    # Look up directly in res_xreg_by_source by stripping the split suffix,
    # or fall back to checking the metrics dict itself (NaN RMSE_winter = no data).
    results = df_metrics_ft_comparison[tgt_label]
    # If all models return NaN for RMSE_winter, there's no winter data
    winter_vals = [
        results[m]["RMSE_winter"] for m in FT_MODEL_ORDER
        if m in results and "RMSE_winter" in results[m]
    ]
    return any(not np.isnan(v) for v in winter_vals)


def get_metrics_for_region(tgt_label):
    """Return only the METRICS rows that are valid for this region."""
    if not has_winter(tgt_label):
        return [(k, l, r) for k, l, r in METRICS if k != "RMSE_winter"]
    return METRICS


n_tgts = len(df_metrics_ft_comparison)
# Use max metric count across regions to size the figure
n_metrics = len(METRICS)

fig, axes = plt.subplots(
    n_metrics,
    n_tgts,
    figsize=(3.5 * n_tgts, 3.8 * n_metrics),
    sharey=False,
)
axes = np.array(axes).reshape(n_metrics, n_tgts)

for col, (tgt_label, results) in enumerate(df_metrics_ft_comparison.items()):
    region_metrics = get_metrics_for_region(tgt_label)
    for row, (metric_key, metric_label, is_r2) in enumerate(METRICS):
        ax = axes[row, col]

        # Hide the axis entirely if this metric isn't valid for this region
        if (metric_key, metric_label, is_r2) not in region_metrics:
            ax.set_visible(False)
            continue

        models = [m for m in FT_MODEL_ORDER if m in results]
        vals = [results[m][metric_key] for m in models]
        colors = [FT_COLORS[m] for m in models]
        display_vals = [max(v, R2_YMIN_CLIP) if is_r2 else v for v in vals]
        valid_vals = [v for v in vals if not np.isnan(v)]
        valid_display = [v for v in display_vals if not np.isnan(v)]

        bars = ax.bar(range(len(models)),
                      display_vals,
                      color=colors,
                      edgecolor="white",
                      linewidth=0.8,
                      alpha=0.75,
                      zorder=3)

        for i, mod in enumerate(models):
            if mod.startswith("foundation"):
                bars[i].set_edgecolor("black")
                bars[i].set_linewidth(1.5)
                bars[i].set_alpha(1.0)

        if is_r2:
            ymin = max(min(valid_display) * 1.15, R2_YMIN_CLIP * 1.05)
            ymax = max(max(valid_display) * 1.15, 0.5)
        else:
            ymin = 0
            ymax = max(valid_vals) * 1.45
        ax.set_ylim(ymin, ymax)

        for bar, val, dval in zip(bars, vals, display_vals):
            clipped = is_r2 and val < R2_YMIN_CLIP
            txt = f"{val:.2f}" + ("*" if clipped else "")
            bar_top = bar.get_height()
            if bar_top > 0.78 * ymax:
                y_pos, va, color = bar_top - 0.06, "top", "white"
            elif bar_top >= 0:
                y_pos, va, color = bar_top + 0.02, "bottom", "black"
            else:
                y_pos, va, color = bar_top - 0.04, "top", "black"
            ax.text(bar.get_x() + bar.get_width() / 2,
                    y_pos,
                    txt,
                    ha="center",
                    va=va,
                    fontsize=6.5,
                    color=color,
                    clip_on=False)

        if is_r2:
            ax.axhline(0,
                       color="black",
                       linewidth=0.8,
                       linestyle="--",
                       zorder=2)
        if is_r2 and any(v < R2_YMIN_CLIP for v in vals):
            ax.text(0.98,
                    0.02,
                    "* clipped",
                    transform=ax.transAxes,
                    ha="right",
                    va="bottom",
                    fontsize=5.5,
                    color="#666666",
                    style="italic")

        ax.set_xticks(range(len(models)))
        ax.set_xticklabels(
            [FT_LABELS[m] for m in models],
            rotation=40,
            ha="right",
            fontsize=max(NATURE_SPECS["font_max_pt"] - 1, 6),
        )

        if col == 0:
            ax.set_ylabel(metric_label, fontsize=NATURE_SPECS["font_max_pt"])
        if row == 0:
            ax.set_title(tgt_label,
                         fontsize=NATURE_SPECS["font_max_pt"] + 1,
                         fontweight="bold",
                         pad=6)

        apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=False)

fig.suptitle(
    f"Foundation vs single-source Transformers — FT strategy: {FT_STRATEGY}",
    fontsize=NATURE_SPECS["font_max_pt"] + 2,
    fontweight="bold",
    y=1.01,
)
fig.tight_layout(rect=[0, 0, 1, 0.99])
fig.savefig(f"figures/foundation_vs_single_source_ft_{FT_STRATEGY}.png",
            dpi=NATURE_SPECS["dpi"],
            bbox_inches="tight")
plt.show()